In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
import copy
import xarray as xr
import os
import re
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

torch.cuda.is_available()

True

In [2]:
data_dir = Path("data_fixed")
metadata_path = data_dir / "station_metadata.csv"

feature_columns = ["rain", "temp", "wetb", "dewpt", "vappr", "rhum", "msl"]
history_hours = 24
horizon_hours = 6
block_length = pd.Timedelta(days=100)
batch_size = 64

station_metadata = pd.read_csv(metadata_path)
station_names = station_metadata["station_name"].astype(str).tolist()

# Load each station, count missing measurement values, then align stations
# on a shared hourly timeline.
station_frames = []
missing_counts = []

for station_name in station_names:
    path = data_dir / f"{station_name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing station file: {path}")

    df = pd.read_csv(
        path,
        usecols=["date"] + feature_columns,
        na_values=["", " ", "-", "NA", "N/A"],
    )

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    for column in feature_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce").astype("float32")

    missing_counts.append(int(df[feature_columns].isna().sum().sum()))

    df = df.set_index("date").sort_index()
    df = df[~df.index.duplicated(keep="first")]
    station_frames.append(df)

station_metadata["missing_data_points"] = missing_counts

panel = pd.concat(
    station_frames,
    axis=1,
    keys=station_names,
).sort_index()

del station_frames

In [3]:
# Reindex to hourly timestamps so missing hours count as missing data.
hourly_index = pd.date_range(
    panel.index.min().ceil("h"),
    panel.index.max().floor("h"),
    freq="h",
)
panel = panel.reindex(hourly_index)

timestamps = panel.index
values = panel.to_numpy(dtype=np.float32, copy=False)

# A time is complete when all selected measurements at all stations exist.
complete = ~np.isnan(values).any(axis=1)

# For each start index i:
# input hours: i through i+23
# prediction target: i+23+6
target_offset = history_hours - 1 + horizon_hours
possible_starts = np.arange(max(0, len(values) - target_offset))

complete_prefix = np.concatenate(
    ([0], np.cumsum(complete, dtype=np.int64))
)
history_complete = (
    complete_prefix[possible_starts + history_hours]
    - complete_prefix[possible_starts]
) == history_hours
target_complete = complete[possible_starts + target_offset]

valid_starts = possible_starts[history_complete & target_complete]
if len(valid_starts) == 0:
    raise ValueError("No complete 24-hour input windows with a valid 6-hour target.")


In [4]:
# Split each 100-day block by target time: 70% train, 15% validation, 15% test.
target_times = timestamps[valid_starts + target_offset]
train_parts, val_parts, test_parts = [], [], []

block_start = target_times.min().normalize()
last_target = target_times.max()

while block_start <= last_target:
    block_end = block_start + block_length
    train_cut = block_start + block_length * 0.70
    val_cut = block_start + block_length * 0.85

    in_block = (target_times >= block_start) & (target_times < block_end)

    train_parts.append(valid_starts[in_block & (target_times < train_cut)])
    val_parts.append(
        valid_starts[
            in_block & (target_times >= train_cut) & (target_times < val_cut)
        ]
    )
    test_parts.append(valid_starts[in_block & (target_times >= val_cut)])

    block_start = block_end

train_starts = np.concatenate(train_parts)
val_starts = np.concatenate(val_parts)
test_starts = np.concatenate(test_parts)


class WeatherWindows(Dataset):
    def __init__(self, values, starts):
        self.values = values
        self.starts = starts

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, index):
        start = self.starts[index]
        target_index = start + target_offset

        # Copy each sample so PyTorch receives writable arrays.
        x = self.values[start : start + history_hours].copy()
        y = self.values[target_index].copy()

        return torch.from_numpy(x), torch.from_numpy(y)


train_dataset = WeatherWindows(values, train_starts)
val_dataset = WeatherWindows(values, val_starts)
test_dataset = WeatherWindows(values, test_starts)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Stations:", len(station_names))
print("Missing measurement values by station:")
display(station_metadata[["station_name", "missing_data_points"]])

print("Train windows:", len(train_dataset))
print("Validation windows:", len(val_dataset))
print("Test windows:", len(test_dataset))
print(
    "Input shape per sample:",
    (history_hours, len(station_names), len(feature_columns)),
)
print("Target shape per sample:", (len(station_names), len(feature_columns)))

# Example batch shapes:
X_batch, Y_batch = next(iter(train_loader))
print("Batch input shape:", X_batch.shape)
print("Batch target shape:", Y_batch.shape)

Stations: 14
Missing measurement values by station:


,station_name,missing_data_points
0,PHOENIX_PARK,0
1,MACE_HEAD,234
2,OAK_PARK,338
3,MOORE_PARK,0
4,SHERKIN_ISLAND,315
5,MULLINGAR,62
6,ROCHES_POINT,193
7,DUNSANY,99
8,GURTEEN,229
9,JOHNSTOWN_CASTLE_2,9664


Train windows: 92309
Validation windows: 20050
Test windows: 19613
Input shape per sample: (24, 14, 7)
Target shape per sample: (14, 7)
Batch input shape: torch.Size([64, 24, 98])
Batch target shape: torch.Size([64, 98])


In [6]:
def make_full_tensors(dataset):
    X = np.stack([
        dataset[i][0].numpy()
        for i in range(len(dataset))
    ])
    Y = np.stack([
        dataset[i][1].numpy()
        for i in range(len(dataset))
    ])
    return torch.from_numpy(X), torch.from_numpy(Y)

X_test, Y_test = make_full_tensors(test_dataset)
X_train, Y_train = make_full_tensors(train_dataset)
feature_mean = X_train.mean(dim=(0, 1, 2), keepdim=True)
feature_std = X_train.std(dim=(0, 1, 2), keepdim=True).clamp_min(1e-6)
y_mean = feature_mean.reshape(1, 1, -1)
y_std = feature_std.reshape(1, 1, -1)



print("Train:", X_train.shape, Y_train.shape)
print("Test:", X_test.shape, Y_test.shape)

Train: torch.Size([92309, 24, 98]) torch.Size([92309, 98])
Test: torch.Size([19613, 24, 98]) torch.Size([19613, 98])


In [7]:
def normalize_batch(X_batch, Y_batch):
    X_batch = (X_batch - feature_mean) / feature_std
    Y_batch = (Y_batch - feature_mean.reshape(1, 1, -1)) / feature_std.reshape(1, 1, -1)
    return X_batch, Y_batch

In [21]:
print(feature_mean.shape, feature_std.shape)

torch.Size([1, 1, 1]) torch.Size([1, 1, 1])


# Persistence

In [ ]:
def persistence_prediction(X):
    return X[:, -1]

X_test_scaled = (X_test - feature_mean) / feature_std
Y_test_scaled = (Y_test - y_mean) / y_std


Y_pred_peristence = persistence_prediction(X_test_scaled)

print("Persistence prediction shape:", Y_pred_peristence.shape)

def mse(y_true, y_pred, axis=None):
    return np.mean((y_true - y_pred) ** 2, axis=axis)

def rmse(y_true, y_pred, axis=None):
    return np.sqrt(mse(y_true, y_pred, axis=axis))

y_test_scaled_np = Y_test_scaled.numpy()
y_pred_persistence_np = Y_pred_peristence.numpy()
mse_persistence = mse(y_test_scaled_np, y_pred_persistence_np, axis = 0)
mse_persistence_mean = np.mean(mse_persistence.reshape(-1, len(station_names), len(feature_columns)), axis=(0, 1))
print(mse_persistence_mean.shape)
for i, feature in enumerate(feature_columns):
    print(f"Feature: {feature}, MSE: {mse_persistence_mean[i]:.6e}")

Persistence prediction shape: torch.Size([19613, 98])
(7,)
